# 📊 Phase 4: Model Evaluation and Validation
### Course: Artificial Intelligence & Machine Learning (24CSE0316) | Continuous Evaluation
**Project: PlacementIQ AI**

---
## 📑 Syllabus Topics Covered:
1. **Train-Test Split:** Why 80/20? Assessing generalizability on unseen data.
2. **Cross-Validation Techniques:** K-Fold vs. Stratified K-Fold to evaluate fold-to-fold variance and prevent data distribution shifts.
3. **Overfitting and Underfitting:** High Bias vs. High Variance diagnostics via Complexity Curves.
4. **Bias-Variance Trade-off:** Mathematical error decomposition:
   $$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \sigma^2$$
5. **Confusion Matrix:** True Positive (TP), True Negative (TN), False Positive (FP - Type I Error), False Negative (FN - Type II Error).
6. **Classification Performance Metrics:** Accuracy, Precision, Recall, F1-Score, and ROC-AUC Curves.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score, validation_curve
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score

sns.set_theme(style="whitegrid", palette="muted")
print("✅ Validation tools ready!")


In [ ]:
train = pd.read_csv('dataset/train.csv')
test = pd.read_csv('dataset/test.csv')

features = ['CGPA', 'Coding_Skills', 'Internships', 'Projects',
            'Communication_Skills', 'Aptitude_Test_Score', 'Soft_Skills_Rating',
            'Certifications', 'Backlogs']
X_train, y_train = train[features], (train['Placement_Status'] == 'Placed').astype(int)
X_test, y_test = test[features], (test['Placement_Status'] == 'Placed').astype(int)

print(f"Train/Test Partitioning: {len(X_train):,} train (80%), {len(X_test):,} test (20%)")


In [ ]:
# 1. Cross-Validation Comparison: K-Fold vs. Stratified K-Fold
model = DecisionTreeClassifier(max_depth=6, random_state=42)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

kf_scores = cross_val_score(model, X_train[:5000], y_train[:5000], cv=kf)
skf_scores = cross_val_score(model, X_train[:5000], y_train[:5000], cv=skf)

print(f"Standard K-Fold Accuracy   : {kf_scores.mean():.4f} ± {kf_scores.std():.4f}")
print(f"Stratified K-Fold Accuracy : {skf_scores.mean():.4f} ± {skf_scores.std():.4f}")


In [ ]:
# 2. Diagnosing Overfitting & Underfitting via Validation Curve
depths = range(1, 16)
train_scores, val_scores = validation_curve(
    DecisionTreeClassifier(random_state=42),
    X_train[:4000], y_train[:4000],
    param_name="max_depth",
    param_range=depths,
    cv=3,
    scoring="accuracy"
)

plt.figure(figsize=(10, 5))
plt.plot(depths, train_scores.mean(axis=1), 'o-', color='crimson', label='Training Score')
plt.plot(depths, val_scores.mean(axis=1), 's-', color='royalblue', label='Validation Score')
plt.axvspan(1, 3, color='yellow', alpha=0.2, label='Underfitting Zone (High Bias)')
plt.axvspan(3, 7, color='green', alpha=0.2, label='Optimal Complexity')
plt.axvspan(7, 15, color='orange', alpha=0.2, label='Overfitting Zone (High Variance)')
plt.title("Overfitting vs. Underfitting: Decision Tree Depth Curve", fontweight='bold')
plt.xlabel("Max Tree Depth")
plt.ylabel("Accuracy Score")
plt.legend()
plt.show()


In [ ]:
# 3. Confusion Matrix Breakdown
rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

labels = [
    [f"TN: {tn:,}\n(Correct Reject)", f"FP: {fp:,}\n(Type I Error)"],
    [f"FN: {fn:,}\n(Type II Error)", f"TP: {tp:,}\n(Correct Placed)"]
]
sns.heatmap(cm, annot=labels, fmt="", cmap="Blues", cbar=False, ax=ax1, annot_kws={"size": 12, "weight": "bold"})
ax1.set_title("Confusion Matrix with Error Types", fontweight='bold')
ax1.set_xlabel("Predicted Label")
ax1.set_ylabel("True Label")

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
ax2.plot(fpr, tpr, color='forestgreen', linewidth=2.5, label=f'Random Forest (AUC = {roc_auc_score(y_test, y_proba):.4f})')
ax2.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax2.set_title("Receiver Operating Characteristic (ROC) Curve", fontweight='bold')
ax2.set_xlabel("False Positive Rate")
ax2.set_ylabel("True Positive Rate")
ax2.legend()

plt.tight_layout()
plt.show()

print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["Not Placed", "Placed"]))
